Обучение на кагле на GPU T4 x2

In [ ]:
!pip install ultralytics

import os
import yaml
from ultralytics import YOLO

target_file = 'layout.yaml'
found_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    if target_file in files:
        found_path = root
        break

if found_path:
    print(f"Датасет найден в папке: {found_path}")
    base_data_path = os.path.join(found_path, "raw")
    
    if not os.path.exists(base_data_path):
        base_data_path = found_path
        print(f"Папка 'raw' не найдена внутри")

    layout = {
        'path': base_data_path,
        'train': 'images/train',
        'val': 'images/val',
        'nc': 10,
        'names': [
            'pedestrian', 'people', 'bicycle', 'car', 'van', 
            'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
        ]
    }

    with open('/kaggle/working/kaggle_layout.yaml', 'w') as f:
        yaml.dump(layout, f)

    print("Конфигурационный файл kaggle_layout.yaml успешно сгенерирован!")

    model = YOLO("yolov8m.pt")
    model.train(
        data='/kaggle/working/kaggle_layout.yaml',
        epochs=40,
        imgsz=1024,
        batch=8,
        amp=False,
        mosaic=0.8,
        close_mosaic=5,
        optimizer='AdamW',
        workers=2,
        name="yolov8m_visdrone_kaggle_fast",
        device=[0, 1]
    )

else:
    print("Файл layout.yaml не найден в /kaggle/input!")
    print(f"Доступные папки в корне /kaggle/input: {os.listdir('/kaggle/input')}")